# Módulo 12. Simulación de escenarios de intervención educativa

El objetivo de este módulo es demostrar el potencial práctico del modelo predictivo desarrollado.

A partir de un municipio real, se construye un escenario hipotético en el que algunos indicadores educativos mejoran como resultado de posibles intervenciones institucionales.

Posteriormente, se compara la predicción del modelo antes y después de dichas mejoras, con el propósito de ilustrar cómo esta herramienta puede apoyar la toma de decisiones y la priorización de políticas públicas.

In [1]:
import pandas as pd
import numpy as np
import joblib

from pathlib import Path

In [2]:
PROJECT_ROOT = Path.cwd().parent

df = pd.read_csv(
    PROJECT_ROOT /
    "data" /
    "processed" /
    "dataset_modelado.csv"
)

modelo = joblib.load(
    PROJECT_ROOT /
    "models" /
    "random_forest_desercion.pkl"
)

print(df.shape)

(15707, 85)


## Selección del municipio de estudio

Como caso de estudio se selecciona el municipio de **La Chorrera (Amazonas)**, ya que presentó una de las mayores tasas de deserción escolar observadas en el año 2024.

Sobre este municipio se evaluará el efecto potencial de mejoras hipotéticas en algunos indicadores educativos.

In [3]:
municipio = df[
    (df["MUNICIPIO"] == "La Chorrera") &
    (df["AÑO"] == 2024)
].copy()

municipio[
    [
        "MUNICIPIO",
        "DEPARTAMENTO",
        "DESERCIÓN",
        "COBERTURA_NETA",
        "APROBACIÓN",
        "REPROBACIÓN",
        "REPITENCIA"
    ]
]

,MUNICIPIO,DEPARTAMENTO,DESERCIÓN,COBERTURA_NETA,APROBACIÓN,REPROBACIÓN,REPITENCIA
32,La Chorrera,Amazonas,14.81,50.38,78.63,6.56,8.85


## Construcción del escenario de intervención

Con el propósito de ilustrar el uso práctico del modelo, se construye un escenario hipotético en el que el municipio mejora algunos de sus indicadores educativos.

Las modificaciones realizadas no corresponden a datos reales, sino a una simulación que representa posibles efectos de políticas públicas orientadas a fortalecer la permanencia escolar.

In [4]:
# Copia del municipio
escenario = municipio.copy()

# Mejoras hipotéticas

escenario["COBERTURA_NETA"] += 5

escenario["APROBACIÓN"] += 3

escenario["REPROBACIÓN"] -= 2

escenario["REPITENCIA"] -= 1

# Evitar porcentajes negativos

escenario["REPROBACIÓN"] = escenario["REPROBACIÓN"].clip(lower=0)

escenario["REPITENCIA"] = escenario["REPITENCIA"].clip(lower=0)

escenario[
    [
        "COBERTURA_NETA",
        "APROBACIÓN",
        "REPROBACIÓN",
        "REPITENCIA"
    ]
]

,COBERTURA_NETA,APROBACIÓN,REPROBACIÓN,REPITENCIA
32,55.38,81.63,4.56,7.85


## Comparación entre el escenario actual y el escenario intervenido

Finalmente, el modelo predictivo se utiliza para estimar la tasa de deserción escolar esperada tanto para la situación actual como para el escenario de intervención.

La diferencia entre ambas predicciones permite ilustrar el posible impacto que tendrían mejoras en algunos indicadores educativos sobre la permanencia escolar.

In [8]:
# ============================================
# Simulación de escenario de intervención
# ============================================

# Seleccionar el municipio
municipio = df[
    (df["MUNICIPIO"] == "La Chorrera") &
    (df["AÑO"] == 2024)
].copy()

# Crear una copia para el escenario intervenido
escenario = municipio.copy()

# -----------------------------
# Mejoras hipotéticas
# -----------------------------

escenario["COBERTURA_NETA"] += 5
escenario["APROBACIÓN"] += 3
escenario["REPROBACIÓN"] -= 2
escenario["REPITENCIA"] -= 1

# Evitar valores negativos
escenario["REPROBACIÓN"] = escenario["REPROBACIÓN"].clip(lower=0)
escenario["REPITENCIA"] = escenario["REPITENCIA"].clip(lower=0)

# -----------------------------
# Preparar variables
# -----------------------------

# Mantener únicamente las variables usadas por el modelo
X_original = municipio.loc[:, modelo.feature_names_in_].copy()
X_intervenido = escenario.loc[:, modelo.feature_names_in_].copy()

# Rellenar valores faltantes utilizando la mediana
# (igual que durante el entrenamiento)
medianas = df[modelo.feature_names_in_].median(numeric_only=True)

X_original = X_original.fillna(medianas)
X_intervenido = X_intervenido.fillna(medianas)

# -----------------------------
# Verificaciones
# -----------------------------

print("Número de variables:", len(modelo.feature_names_in_))
print("Columnas coinciden:",
      list(X_original.columns) == list(modelo.feature_names_in_))

print("\nValores faltantes en X_original:")
print(X_original.isna().sum().sum())

print("\nValores faltantes en X_intervenido:")
print(X_intervenido.isna().sum().sum())

# -----------------------------
# Predicciones
# -----------------------------

pred_original = modelo.predict(X_original)[0]
pred_intervenido = modelo.predict(X_intervenido)[0]

# -----------------------------
# Resultados
# -----------------------------

print("\n==============================")
print("RESULTADOS DE LA SIMULACIÓN")
print("==============================")

print(f"Municipio: {municipio.iloc[0]['MUNICIPIO']}")
print(f"Departamento: {municipio.iloc[0]['DEPARTAMENTO']}")

print(f"\nDeserción real 2024       : {municipio.iloc[0]['DESERCIÓN']:.2f}%")
print(f"Predicción actual         : {pred_original:.2f}%")
print(f"Predicción intervenida    : {pred_intervenido:.2f}%")
print(f"Reducción estimada        : {pred_original - pred_intervenido:.2f} puntos porcentuales")

Número de variables: 69
Columnas coinciden: True

Valores faltantes en X_original:
0

Valores faltantes en X_intervenido:
0

RESULTADOS DE LA SIMULACIÓN
Municipio: La Chorrera
Departamento: Amazonas

Deserción real 2024       : 14.81%
Predicción actual         : 13.78%
Predicción intervenida    : 11.31%
Reducción estimada        : 2.47 puntos porcentuales


## Interpretación del escenario de intervención

El escenario simulado muestra que, al mejorar algunos indicadores educativos como la cobertura neta, la aprobación y reducir la reprobación y la repitencia, el modelo estima una disminución importante en la tasa de deserción escolar.

En el caso del municipio de La Chorrera (Amazonas), la predicción pasó de aproximadamente **13.78%** a **11.31%**, lo que representa una reducción estimada cercana a **2.47 puntos porcentuales**.

Aunque esta simulación no demuestra una relación causal, sí evidencia el potencial del modelo como herramienta para apoyar la planeación de políticas públicas, permitiendo evaluar de manera prospectiva diferentes escenarios de intervención educativa.